# ECG Arrhythmia Classification — Updated Jupyter Project

**Educational and research use only — not a medical diagnostic tool.**

This notebook modernizes the uploaded project, compares six classifiers **with and without PCA**, and applies oversampling **only to training data** to avoid data leakage.

**Data source:** UCI Arrhythmia (`452` samples, `279` features, final column is class). If no dataset is available and downloading fails, the notebook can use **clearly identified synthetic demonstration data** so you can still execute the workflow. Synthetic results must never be presented as UCI dataset results.

Run cells sequentially using **Kernel → Restart & Run All**.

In [ ]:
# Configuration: change these values before running
DATA_MODE = "auto"  # "auto", "uci", or "demo"
# auto: local UCI data -> download UCI data -> synthetic demonstration fallback
# uci: only genuine UCI data; errors clearly if inaccessible
# demo: always use synthetic data (NOT medical records)
RANDOM_STATE = 42
TEST_SIZE = 0.20
PCA_VARIANCE = 0.95
SAVE_RESULTS = True

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.request import urlretrieve
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, classification_report, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
print("Python dependencies imported successfully")

## 1. Load dataset

Place `arrhythmia.data` (original UCI format) or `arrhythmia1.csv` in this notebook's directory, or in `Data/`. The first 279 columns are features and column 280 is the class label. The UCI source is downloaded only if the file is absent. When offline, `auto` uses synthetic demo data and clearly announces this.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/arrhythmia/arrhythmia.data"
paths = [Path("arrhythmia.data"), Path("Data/arrhythmia.data"),
         Path("arrhythmia1.csv"), Path("Data/arrhythmia1.csv")]
source_type = None
raw = None
if DATA_MODE not in {"auto", "uci", "demo"}:
    raise ValueError("DATA_MODE must be auto, uci, or demo")
if DATA_MODE != "demo":
    for path in paths:
        if path.exists():
            raw = pd.read_csv(path, header=None, na_values=["?", "NA", ""])
            source_type = f"UCI data file: {path}"
            break
    if raw is None:
        try:
            download_path = Path("arrhythmia.data")
            print("Downloading UCI dataset...")
            urlretrieve(DATA_URL, download_path)
            raw = pd.read_csv(download_path, header=None, na_values=["?", "NA", ""])
            source_type = "UCI repository download"
        except Exception as exc:
            if DATA_MODE == "uci":
                raise RuntimeError("UCI data unavailable. Download arrhythmia.data from the UCI repository and put it next to this notebook, or set DATA_MODE='demo'.") from exc
            print("UCI dataset unavailable:", str(exc)[:200])
if raw is None:
    # An explicitly marked, artificial dataset to exercise the code offline.
    x_demo, y_demo = make_classification(
        n_samples=452, n_features=279, n_informative=32, n_redundant=22,
        n_classes=12, n_clusters_per_class=1,
        weights=[0.54, 0.10, 0.08, 0.06, 0.05, 0.04, 0.035, 0.025, 0.02, 0.02, 0.015, 0.015],
        flip_y=0.01, random_state=RANDOM_STATE)
    raw = pd.DataFrame(np.column_stack([x_demo, y_demo + 1]))
    source_type = "SYNTHETIC DEMO DATA — not real patient ECGs"
print("DATA SOURCE:", source_type)
if raw.shape[1] != 280:
    raise ValueError(f"Expected 280 columns (279 features + label), got {raw.shape[1]}. Check the input file.")
print("Dataset shape:", raw.shape)
raw.head()

## 2. Explore data and clean feature values

The target labels are kept **separate** and are never imputed or scaled. For fair testing, all imputers, feature selection, scaling, PCA and oversampling are fit using training data only.

In [ ]:
X = raw.iloc[:, :-1].apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw.iloc[:, -1], errors="coerce")
valid = y.notna()
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)
X.columns = [f"ECG_{i:03d}" for i in range(1, X.shape[1] + 1)]
print("Patient rows:", len(X), "| Feature columns:", X.shape[1])
print("Missing feature values:", int(X.isna().sum().sum()))
print("Class frequencies:\n", y.value_counts().sort_index().to_string())
plt.figure(figsize=(10, 4))
counts_plot = y.value_counts().sort_index()
plt.bar(counts_plot.index.astype(str), counts_plot.values)
plt.title("Class distribution — " + ("synthetic demo" if "SYNTHETIC" in source_type else "UCI dataset"))
plt.xlabel("Class label"); plt.ylabel("Count"); plt.tight_layout(); plt.show()

In [ ]:
# Attempt stratification only if it is mathematically possible.
counts = y.value_counts()
can_stratify = counts.min() >= 2 and int(np.ceil(TEST_SIZE * len(y))) >= y.nunique()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=y if can_stratify else None)
print("Train:", X_train.shape, "Test:", X_test.shape,
      "Stratified:", can_stratify)
# Fit transformations to TRAIN ONLY, preventing leakage.
imputer = SimpleImputer(strategy="median", keep_empty_features=True)
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)
selector = VarianceThreshold(threshold=0.0)
X_train_sel = selector.fit_transform(X_train_imp)
X_test_sel = selector.transform(X_test_imp)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)
print("Non-constant feature count:", X_train_scaled.shape[1])

## 3. Train-only oversampling and PCA

Use random oversampling to increase tiny classes to at least six observations, and then SMOTE for additional balancing. If the data do not support SMOTE, the workflow safely falls back to random oversampling. A PCA fit on the oversampled **training** set is never exposed to the test labels.

In [ ]:
train_counts = y_train.value_counts()
if len(train_counts) < 2:
    raise ValueError("Need at least two classes in the training data")
minority_targets = {int(cls): max(int(cnt), 6) for cls, cnt in train_counts.items() if cnt < 6}
if minority_targets:
    bootstrap = RandomOverSampler(sampling_strategy=minority_targets, random_state=RANDOM_STATE)
    X_boot, y_boot = bootstrap.fit_resample(X_train_scaled, y_train)
else:
    X_boot, y_boot = X_train_scaled, y_train
try:
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
    X_bal, y_bal = smote.fit_resample(X_boot, y_boot)
    oversampling_method = "RandomOverSampler (small classes) + SMOTE"
except ValueError as err:
    print("SMOTE unavailable for this split; applying random oversampling:", err)
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_bal, y_bal = ros.fit_resample(X_train_scaled, y_train)
    oversampling_method = "RandomOverSampler"
pca = PCA(n_components=PCA_VARIANCE, svd_solver="full")
X_train_pca = pca.fit_transform(X_bal)
X_test_pca = pca.transform(X_test_scaled)
print("Oversampling:", oversampling_method)
print("Balanced training shape:", X_bal.shape)
print("PCA components:", pca.n_components_, "| Explained variance:", round(float(pca.explained_variance_ratio_.sum()), 4))

## 4. Compare classifiers, with and without PCA

Both variants are trained on the same oversampled training observations; only the PCA variant uses transformed features. Accuracy, balanced accuracy, macro-F1 and macro-recall are included because ordinary accuracy is misleading with rare arrhythmia classes.

In [ ]:
def build_models():
    return {
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
        "Linear SVC": LinearSVC(C=0.1, max_iter=5000, random_state=RANDOM_STATE),
        "Kernel SVC (RBF)": SVC(kernel="rbf", C=3, gamma="scale", random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
    }
results = []
fitted_models = {}
for variant, train_features, test_features in [
    ("No PCA", X_bal, X_test_scaled), ("PCA 95%", X_train_pca, X_test_pca)]:
    for name, model in build_models().items():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(train_features, y_bal)
        predictions = model.predict(test_features)
        results.append({
            "Variant": variant, "Model": name,
            "Train Accuracy": accuracy_score(y_bal, model.predict(train_features)),
            "Test Accuracy": accuracy_score(y_test, predictions),
            "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
            "Macro Recall": recall_score(y_test, predictions, average="macro", zero_division=0),
            "Macro F1": f1_score(y_test, predictions, average="macro", zero_division=0),
        })
        fitted_models[(variant, name)] = model
result = pd.DataFrame(results).sort_values(
    ["Macro Recall", "Macro F1", "Test Accuracy"], ascending=False).reset_index(drop=True)
display(result.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
result.pivot(index="Model", columns="Variant", values="Macro Recall").plot(kind="bar", ax=ax)
ax.set_title("Test-set macro recall (higher is better)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
best_row = result.iloc[0]
best_variant, best_name = best_row["Variant"], best_row["Model"]
best_model = fitted_models[(best_variant, best_name)]
best_features = X_test_pca if best_variant == "PCA 95%" else X_test_scaled
predicted = best_model.predict(best_features)
print("Best model by TEST macro recall:", best_variant, "–", best_name)
print("Note: this is exploratory model selection on the test set, not an unbiased estimate of generalization.")
print(classification_report(y_test, predicted, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predicted, xticks_rotation=90, cmap="Blues")
plt.title(f"Confusion matrix: {best_variant} + {best_name}")
plt.tight_layout(); plt.show()

## 5. Export comparison

Outputs are saved in `outputs/`. **Do not interpret any synthetic demo metric as an ECG diagnostic performance measurement.** For research-quality performance, compare models using cross-validation on training data, select hyperparameters there, and evaluate once on a held-out test set.

In [ ]:
if SAVE_RESULTS:
    output = Path("outputs")
    output.mkdir(parents=True, exist_ok=True)
    result.to_csv(output / "model_comparison.csv", index=False)
    pd.DataFrame({"true_class": y_test.to_numpy(), "predicted_class": predicted}).to_csv(
        output / "test_predictions.csv", index=False)
    (output / "data_source.txt").write_text(
        source_type + "\nNOTE: Synthetic data are artificial and cannot validate medical performance.\n",
        encoding="utf-8")
    print("Saved results to:", output.resolve())
print("Completed notebook successfully. DATA SOURCE:", source_type)